In [ ]:
import os
import zipfile
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import librosa
import librosa.display
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# =======================
# 1. CONFIGURACIÓN
# =======================
DATASET_PATH = "./AUDIOS(1000)"
OUTPUT_DIR = "spectrograms"
TARGET_SAMPLE_RATE = 22000
AUDIO_DURATION = 1.0
TARGET_LENGTH = int(TARGET_SAMPLE_RATE * AUDIO_DURATION)
N_MELS = 128
SPECTROGRAM_SHAPE = (N_MELS, 44)
IMAGE_SIZE = (128, 44)
BATCH_SIZE = 32
TRAIN_SPLIT = 0.8  # 80% entrenamiento, 20% validación
PCA_COMPONENTS = 100  # Número de componentes para la reducción dimensional
EPOCHS = 10

# =======================
# 2. FUNCIONES PARA PROCESAR AUDIO Y GENERAR ESPECTROGRAMAS
# =======================
def load_audio(file_path, sr=TARGET_SAMPLE_RATE):
    """Carga un archivo de audio y ajusta su longitud"""
    audio, _ = librosa.load(file_path, sr=sr)
    if len(audio) < TARGET_LENGTH:
        padding = TARGET_LENGTH - len(audio)
        audio = np.pad(audio, (0, padding), mode='constant')
    else:
        audio = audio[:TARGET_LENGTH]
    return audio

def audio_to_melspectrogram(audio, sr=TARGET_SAMPLE_RATE, n_mels=N_MELS):
    """Convierte audio a espectrograma mel"""
    spectrogram = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    spectrogram_db = librosa.power_to_db(spectrogram, ref=np.max)
    return spectrogram_db

def save_spectrogram_image(spectrogram, label, is_test, idx):
    """Guarda el espectrograma como imagen"""
    subset = "validation" if is_test else "train"
    label_dir = os.path.join(OUTPUT_DIR, subset, label)
    os.makedirs(label_dir, exist_ok=True)
    filename = f"{label}_{idx}.jpg"
    filepath = os.path.join(label_dir, filename)

    # Normaliza el espectrograma para la visualización
    img = (spectrogram - spectrogram.min()) / (spectrogram.max() - spectrogram.min())
    img = (img * 255).astype(np.uint8)
    im = Image.fromarray(img)
    im.convert("L").save(filepath)
    return im, filepath

# =======================
# 3. EXTRACCIÓN DE ARCHIVOS ZIP
# =======================
def extract_zip(zip_path, extract_path):
    """Extrae archivos de un zip"""
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"✅ Archivo '{zip_path}' descomprimido en '{extract_path}'.")

# =======================
# 4. PROCESAMIENTO PRINCIPAL
# =======================
def process_audio_files():
    """Procesa los archivos de audio y genera espectrogramas"""
    # Asegúrate de que existan los directorios de salida
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "train"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "validation"), exist_ok=True)

    # Encuentra todos los archivos MP3 (limitamos a 1000)
    all_files = []
    for root, dirs, files in os.walk(DATASET_PATH):
        for file in files:
            if file.lower().endswith(".mp3"):
                file_path = os.path.join(root, file)
                label = os.path.basename(os.path.dirname(file_path))
                all_files.append((file_path, label))
                if len(all_files) >= 1000:  # Limitamos a exactamente 1000 archivos
                    break
        if len(all_files) >= 1000:
            break
    print(f"🎷 Archivos MP3 encontrados: {len(all_files)}")

    # Mezcla y divide en conjuntos de entrenamiento (80%) y validación (20%)
    random.shuffle(all_files)
    split_idx = int(len(all_files) * TRAIN_SPLIT)
    train_files = all_files[:split_idx]
    validation_files = all_files[split_idx:]

    print(f"📊 Dividido en {len(train_files)} archivos de entrenamiento y {len(validation_files)} de validación")

    # Procesa archivos de entrenamiento (originales)
    print("💽 Procesando archivos de entrenamiento originales...")
    train_count = 0
    for idx, (file_path, label) in enumerate(train_files):
        audio = load_audio(file_path)
        spec = audio_to_melspectrogram(audio)
        spec = np.pad(spec, ((0,0), (0, max(0, SPECTROGRAM_SHAPE[1] - spec.shape[1]))), mode='constant')[:, :SPECTROGRAM_SHAPE[1]]
        im, _ = save_spectrogram_image(spec, label, is_test=False, idx=train_count)
        train_count += 1

    # Aumentación sintética de datos de entrenamiento
    print("🔄 Realizando aumento sintético de datos...")
    augmented_count = train_count
    target_count = 8000  # Queremos llegar a 5000 muestras totales para entrenamiento

    # Calculamos cuántas aumentaciones necesitamos por archivo original
    augs_per_file = int((target_count - train_count) / train_count) + 1
    print(f"⚙️ Generando {augs_per_file} aumentaciones por archivo, objetivo: {target_count} muestras")

    # Recorremos todas las clases en el directorio de entrenamiento
    train_dir = os.path.join(OUTPUT_DIR, "train")
    for class_label in os.listdir(train_dir):
        class_dir = os.path.join(train_dir, class_label)
        if not os.path.isdir(class_dir):
            continue

        # Obtenemos la lista de archivos originales para esta clase
        original_files = [f for f in os.listdir(class_dir) if not "aug_" in f and f.endswith(".jpg")]

        # Aumentamos cada imagen original
        for img_file in original_files:
            # Cargamos la imagen original
            orig_path = os.path.join(class_dir, img_file)
            im = Image.open(orig_path).convert('L')

            # Generamos diferentes tipos de aumentaciones
            augmentations = []

            # Rotaciones
            for angle in [5, -5, 10, -10, 15, -15, 20, -20]:
                augmentations.append(im.rotate(angle))

            # Cambios de brillo
            for factor in [0.7, 0.8, 0.9, 1.1, 1.2, 1.3]:
                enhancer = ImageEnhance.Brightness(im)
                augmentations.append(enhancer.enhance(factor))

            # Volteo horizontal usando numpy
            img_array = np.array(im)
            flipped_array = np.fliplr(img_array)
            flipped = Image.fromarray(flipped_array)

            augmentations.append(flipped)

            # Combinaciones de volteo con rotación y brillo
            augmentations.append(flipped.rotate(10))
            augmentations.append(flipped.rotate(-10))

            enhancer = ImageEnhance.Brightness(flipped)
            augmentations.append(enhancer.enhance(0.8))
            augmentations.append(enhancer.enhance(1.2))

            # Mezcla y selecciona aumentaciones para no exceder el total objetivo
            random.shuffle(augmentations)
            selected_augs = augmentations[:min(augs_per_file, len(augmentations))]

            # Guarda las aumentaciones seleccionadas
            for aug in selected_augs:
                if augmented_count < target_count:
                    aug_path = os.path.join(class_dir, f"{class_label}_aug_{augmented_count}.jpg")
                    aug.save(aug_path)
                    augmented_count += 1
                else:
                    break

        # Detenemos el proceso si alcanzamos el objetivo
        if augmented_count >= target_count:
            break

    print(f"✅ Aumentación de datos completada: {augmented_count} muestras de entrenamiento generadas")

    # Procesa archivos de validación (solo originales, sin aumentación)
    print("💽 Procesando archivos de validación...")
    for idx, (file_path, label) in enumerate(validation_files):
        audio = load_audio(file_path)
        spec = audio_to_melspectrogram(audio)
        spec = np.pad(spec, ((0,0), (0, max(0, SPECTROGRAM_SHAPE[1] - spec.shape[1]))), mode='constant')[:, :SPECTROGRAM_SHAPE[1]]
        save_spectrogram_image(spec, label, is_test=True, idx=idx)

    print("✅ Espectrogramas guardados en la carpeta 'spectrograms/'")
    return train_files, validation_files

# =======================
# 5. LIMPIEZA Y SINCRONIZACIÓN DE CLASES
# =======================
def clean_and_sync_classes():
    """Elimina carpetas vacías y asegura que las clases estén sincronizadas entre entrenamiento y validación"""
    for subset in ['train', 'validation']:
        subset_path = os.path.join(OUTPUT_DIR, subset)
        for class_dir in os.listdir(subset_path):
            class_path = os.path.join(subset_path, class_dir)
            if os.path.isdir(class_path) and len(os.listdir(class_path)) == 0:
                print(f"⚠️ Eliminando carpeta vacía: {class_path}")
                shutil.rmtree(class_path)

    # Asegúrate de que las mismas clases existan en ambos conjuntos
    train_classes = set(os.listdir(os.path.join(OUTPUT_DIR, "train")))
    val_classes = set(os.listdir(os.path.join(OUTPUT_DIR, "validation")))

    # Elimina clases de validación que no están en entrenamiento
    for cls in val_classes - train_classes:
        shutil.rmtree(os.path.join(OUTPUT_DIR, "validation", cls))

    # Elimina clases de entrenamiento que no están en validación
    for cls in train_classes - val_classes:
        shutil.rmtree(os.path.join(OUTPUT_DIR, "train", cls))

# =======================
# 6. CARGA DE IMÁGENES PARA EL MODELO
# =======================
class DataLoader:
    """Clase para cargar imágenes y preparar datos para el modelo"""
    def __init__(self, directory):
        self.directory = directory
        self.class_indices = {}
        self.filenames = []
        self.labels = []
        self._load_data()

    def _load_data(self):
        """Carga las imágenes y etiquetas"""
        class_dirs = sorted([d for d in os.listdir(self.directory)
                             if os.path.isdir(os.path.join(self.directory, d))])

        self.class_indices = {class_name: i for i, class_name in enumerate(class_dirs)}

        for class_name in class_dirs:
            class_path = os.path.join(self.directory, class_name)
            class_idx = self.class_indices[class_name]

            for filename in os.listdir(class_path):
                if filename.lower().endswith('.jpg'):
                    self.filenames.append(os.path.join(class_path, filename))
                    self.labels.append(class_idx)

    def get_data(self):
        """Devuelve las imágenes en formato 4D (samples, height, width, channels)"""
        features = []
        for filename in self.filenames:
            img = plt.imread(filename)
            img = img.astype('float32') / 255.0  # Normalizar a [0,1]
            img = np.expand_dims(img, axis=-1)    # Agregar canal de color
            features.append(img)

        return np.array(features), np.array(self.labels)

# =======================
# 7. REDUCCIÓN DIMENSIONAL
# =======================
def apply_dimensionality_reduction(X_train, y_train, n_components=PCA_COMPONENTS):
    """Aplica reducción dimensional a los datos de entrenamiento usando PCA"""
    print(f"🔍 Aplicando PCA para reducción dimensional a {n_components} componentes...")

    # Normalizar los datos para PCA (media 0, varianza 1)
    X_mean = np.mean(X_train, axis=0)
    X_std = np.std(X_train, axis=0)
    X_scaled = (X_train - X_mean) / (X_std + 1e-10)  # Evita división por cero

    # Aplicar PCA
    pca = PCA(n_components=n_components)
    X_train_reduced = pca.fit_transform(X_scaled)

    # Calcular varianza explicada
    explained_variance = sum(pca.explained_variance_ratio_)
    print(f"✅ PCA completado - Varianza explicada: {explained_variance:.2%}")

    # Visualizar componentes principales
    plt.figure(figsize=(10, 4))
    plt.plot(np.cumsum(pca.explained_variance_ratio_), 'o-')
    plt.title('Varianza explicada acumulada por componentes')
    plt.xlabel('Número de componentes')
    plt.ylabel('Varianza explicada acumulada')
    plt.grid(True)
    plt.savefig('pca_variance_explained.png')

    return X_train_reduced, pca, X_mean, X_std

def transform_validation_data(X_val, pca, X_mean, X_std):
    """Aplica la misma transformación PCA a los datos de validación"""
    # Normalizar los datos con la misma media y desviación estándar del conjunto de entrenamiento
    X_val_scaled = (X_val - X_mean) / (X_std + 1e-10)

    # Aplicar transformación PCA
    X_val_reduced = pca.transform(X_val_scaled)

    return X_val_reduced

# =======================
# 8. DEFINIR Y ENTRENAR MODELO (MODIFICADO)
# =======================
def build_cnn_model(input_shape, num_classes):
    """Construye modelo CNN"""
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def train_cnn_model(X_train, y_train, X_val, y_val, class_names, fold=None):
    """Entrena modelo CNN con early stopping y guarda gráficas por fold"""
    input_shape = (IMAGE_SIZE[0], IMAGE_SIZE[1], 1)
    num_classes = len(class_names)

    model = build_cnn_model(input_shape, num_classes)

    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=1
    )

    # Graficar curvas de aprendizaje CON NOMBRE DEL FOLD
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train')
    plt.plot(history.history['val_accuracy'], label='Validation')
    plt.title(f'Precisión por época (Fold {fold})' if fold else 'Precisión por época')
    plt.xlabel('Épocas')
    plt.ylabel('Precisión')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Validation')
    plt.title(f'Pérdida por época (Fold {fold})' if fold else 'Pérdida por época')
    plt.xlabel('Épocas')
    plt.ylabel('Pérdida')
    plt.legend()

    plt.tight_layout()
    plt.savefig(f'learning_curves_fold_{fold}.png')  # 👈 Nombre único por fold
    plt.close()

    return model, history

# =======================
# 9. EVALUACIÓN (MODIFICADA)
# =======================
def evaluate_model(model, X_test, y_test, class_names, fold=None):
    """Evalúa el modelo y genera reportes con gráficas por fold"""
    y_pred = np.argmax(model.predict(X_test), axis=1)

    # Matriz de confusión CON NOMBRE DEL FOLD
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    plt.imshow(conf_matrix, interpolation='nearest', cmap='Blues')
    plt.title(f'Matriz de Confusión (Fold {fold})' if fold else 'Matriz de Confusión')
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    thresh = conf_matrix.max() / 2.
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            plt.text(j, i, format(conf_matrix[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if conf_matrix[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Etiqueta Predicha')
    plt.savefig(f'confusion_matrix_fold_{fold}.png')  # 👈 Nombre único por fold
    plt.close()

    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=class_names)

    return accuracy, conf_matrix, report

# =======================
# 10. VALIDACIÓN CRUZADA (MODIFICADA)
# =======================
def train_with_cross_validation(X_train, y_train, X_val, y_val, class_names, n_folds=5):
    """Entrena y evalúa modelos CNN con validación cruzada"""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    fold_accuracies = []
    fold_matrices = []
    fold_models = []

    for fold, (train_idx, _) in enumerate(kf.split(X_train)):
        fold_num = fold + 1
        print(f"\n🔍 Procesando Fold {fold_num}/{n_folds}")

        X_train_fold = X_train[train_idx]
        y_train_fold = y_train[train_idx]

        # Entrenar y obtener curvas de aprendizaje
        model, history = train_cnn_model(
            X_train_fold, y_train_fold,
            X_val, y_val,
            class_names,
            fold=fold_num  # 👈 Pasamos el número de fold
        )

        # Evaluar y generar matriz de confusión
        accuracy, conf_matrix, _ = evaluate_model(
            model, X_val, y_val, class_names,
            fold=fold_num  # 👈 Pasamos el número de fold
        )

        fold_accuracies.append(accuracy)
        fold_matrices.append(conf_matrix)
        fold_models.append(model)

        print(f"📊 Fold {fold_num} completado. Precisión: {accuracy:.2%}")

    # Resultados finales
    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(fold_accuracies)

    print("\n🏆 Resultados de validación cruzada:")
    print(f"Precisión promedio: {mean_accuracy:.2%} ± {std_accuracy:.2%}")
    for i, acc in enumerate(fold_accuracies):
        print(f"  Fold {i+1}: {acc:.2%}")

    # Mejor modelo
    best_idx = np.argmax(fold_accuracies)
    best_model = fold_models[best_idx]
    print(f"\n🥇 Mejor modelo: Fold {best_idx+1} con precisión {fold_accuracies[best_idx]:.2%}")

    return best_model, fold_accuracies, fold_matrices


# =======================
# 12. MATRIZ DE CONFUSIÓN PROMEDIO
# =======================
def plot_average_confusion_matrix(fold_matrices, class_names):
    """Calcula y grafica la matriz de confusión promedio de todos los folds"""
    # Convertir a array numpy para operaciones matriciales
    matrices = np.array(fold_matrices)

    # Calcular promedio a través de los folds
    avg_matrix = np.mean(matrices, axis=0)

    # Configurar figura
    plt.figure(figsize=(10, 8))
    plt.imshow(avg_matrix, interpolation='nearest', cmap='Blues')
    plt.title('Matriz de Confusión Promedio (5 Folds)')
    plt.colorbar()

    # Configurar ejes
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    # Agregar valores numéricos
    thresh = avg_matrix.max() / 2.
    for i in range(avg_matrix.shape[0]):
        for j in range(avg_matrix.shape[1]):
            plt.text(j, i, f"{avg_matrix[i, j]:.1f}",
                    ha="center", va="center",
                    color="white" if avg_matrix[i, j] > thresh else "black",
                    fontsize=8)

    plt.tight_layout()
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Etiqueta Predicha')
    plt.savefig('average_confusion_matrix.png')
    plt.close()
    print("✅ Matriz de confusión promedio guardada en average_confusion_matrix.png")




# =======================
# 11. FUNCIÓN PRINCIPAL (MODIFICADA)
# =======================

def main():
    # Paso 0: Descomprimir el dataset (¡NUEVO!)
    zip_path = "AUDIOS(1000).zip"  # 👈 Reemplaza con tu ruta real
    extract_zip(zip_path, DATASET_PATH)

    # Paso 1: Generar los espectrogramas
    print("🚀 Procesando archivos de audio...")
    train_files, validation_files = process_audio_files()  # 👈 Capturamos los archivos procesados
    clean_and_sync_classes()

    # Validación crítica (¡NUEVO!)
    if len(train_files) == 0:
        raise ValueError("❌ No se encontraron archivos MP3 en el dataset. Verifica la ruta o el ZIP.")

    # Paso 2: Cargar datos
    print("\n📂 Cargando imágenes...")
    train_loader = DataLoader(os.path.join(OUTPUT_DIR, "train"))
    X_train, y_train = train_loader.get_data()

    val_loader = DataLoader(os.path.join(OUTPUT_DIR, "validation"))
    X_val, y_val = val_loader.get_data()

    class_names = list(train_loader.class_indices.keys())
    print(f"🏷️ Clases detectadas: {class_names}")
    print(f"📊 Datos entrenamiento: {X_train.shape}, Validación: {X_val.shape}")

    # Paso 3: Entrenar modelo
    print("\n🎓 Entrenando modelo...")
    best_model, fold_accuracies, _ = train_with_cross_validation(
        X_train, y_train, X_val, y_val, class_names
    )

    print("\n📊 Calculando matriz de confusión promedio...")
    plot_average_confusion_matrix(fold_matrices, class_names)

    # Guardar modelo
    best_model.save('best_cnn_model.h5')
    print("\n✅ Modelo CNN guardado como 'best_cnn_model.h5'")

if __name__ == "__main__":
    main()

✅ Archivo 'AUDIOS(1000).zip' descomprimido en './AUDIOS(1000)'.
🚀 Procesando archivos de audio...
🎷 Archivos MP3 encontrados: 1000
📊 Dividido en 800 archivos de entrenamiento y 200 de validación
💽 Procesando archivos de entrenamiento originales...


KeyboardInterrupt: 